**We continue with the same Used Cars Dataset**

In [ ]:
import pandas as pd

df = pd.read_csv('used_cars_dataset.csv')
print(df.head())

   Car_ID       Brand    Model  Year  Age_Years    Price  Kilometers_Driven  \
0       1         BMW       X5  2019          7  7134000              86551   
1       2         Kia  Picanto  2025          1  4025000               8577   
2       3     Hyundai   Santro  2011         15   300000             177666   
3       4       Honda   Accord  2008         18   300000             217370   
4       5  Volkswagen     Golf  2013         13   342000             145065   

  Fuel_Type Transmission            Owner_Type  Mileage_kmpl  Engine_CC  \
0    Petrol       Manual           Third Owner          16.4       2500   
1    Petrol       Manual  Fourth & Above Owner          11.3       1300   
2    Diesel       Manual           First Owner          14.9       1300   
3    Petrol       Manual           First Owner          12.6       1200   
4    Petrol       Manual           First Owner          18.0       1300   

   Power_BHP  Seats  Color        City  
0      190.1      5  Green  Faisa

we identified Owner as an ordinal categorical column — the best candidate in this dataset for Label Encoding.

**Why Can't We Feed Owner Directly Into a Model?**

If we try to train, for example, a Linear Regression model to predict Price_Lakh using Owner as-is:

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(df[['Owner']], df['Price_Lakh'])   # This will throw an ERROR

KeyError: "None of [Index(['Owner'], dtype='object')] are in the [columns]"

Explanation:

Scikit-learn models perform mathematical operations (matrix multiplication, gradient descent, etc.) internally. Text strings like 'First' cannot participate in these calculations. This is exactly why we need Label Encoding — to convert 'First', 'Second', 'Third' into 0, 1, 2.

**Method 1: Using Scikit-learn's LabelEncoder**

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Step 1: Create the encoder object
le = LabelEncoder()

# Step 2: Fit and transform the 'Owner' column
df['Owner_Encoded'] = le.fit_transform(df['Owner_Type'])

print(df[['Owner_Type', 'Owner_Encoded']])

                Owner_Type  Owner_Encoded
0              Third Owner              3
1     Fourth & Above Owner              1
2              First Owner              0
3              First Owner              0
4              First Owner              0
...                    ...            ...
1995  Fourth & Above Owner              1
1996          Second Owner              2
1997          Second Owner              2
1998           First Owner              0
1999           First Owner              0

[2000 rows x 2 columns]


**Line-by-line explanation:**

LabelEncoder() — creates an instance of the encoder class from sklearn.preprocessing.
.fit_transform(df['Owner']) does two things in one step:

fit: scans the column, finds all unique categories, and sorts them alphabetically by default.

transform: replaces each category with its corresponding integer index (0, 1, 2, ...).

We store the result in a new column Owner_Encoded, keeping the original Owner column for reference — this is good practice so you can always trace back to the original label.

**Checking the Mapping**

In [ ]:
mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print(mapping)

{'First Owner': np.int64(0), 'Fourth & Above Owner': np.int64(1), 'Second Owner': np.int64(2), 'Third Owner': np.int64(3)}


**Method 2 (Recommended): Manual Ordinal Mapping**

In [ ]:
# Step 1: Define the correct logical order
owner_mapping = {
    'First': 0,
    'Second': 1,
    'Third': 2,
    'Fourth & Above': 3
}

# Step 2: Apply the mapping
df['Owner_Encoded'] = df['Owner_Type'].map(owner_mapping)

print(df[['Owner_Type', 'Owner_Encoded']])

**Encoding the Target Column: Transmission for a Classification Task**

Suppose instead of predicting Price_Lakh, our task is to classify whether a car is Automatic or Manual based on other features. Here, Transmission becomes our target/label column:

In [ ]:
from sklearn.preprocessing import LabelEncoder

le_target = LabelEncoder()
df['Transmission_Encoded'] = le_target.fit_transform(df['Transmission'])

print(df[['Transmission', 'Transmission_Encoded']])
print(dict(zip(le_target.classes_, le_target.transform(le_target.classes_))))

     Transmission  Transmission_Encoded
0          Manual                     1
1          Manual                     1
2          Manual                     1
3          Manual                     1
4          Manual                     1
...           ...                   ...
1995    Automatic                     0
1996       Manual                     1
1997    Automatic                     0
1998       Manual                     1
1999    Automatic                     0

[2000 rows x 2 columns]
{'Automatic': np.int64(0), 'Manual': np.int64(1)}


**Teaching Point:** Even though Transmission is technically nominal (Automatic isn't "more" than Manual),

Label Encoding is still perfectly acceptable here because:

It's a binary target column for classification (only 2 classes — 0/1).
Classification algorithms treat the target as class labels, not as a magnitude to be mathematically compared — so there's no "false order" problem for target columns.

This distinguishes Label Encoding on input features (where ordinality matters) from Label Encoding on target/output columns (where it's just an ID for the class).

**Why We Should NOT Label Encode Brand or Fuel_Type**

In [ ]:
le_brand = LabelEncoder()
df['Brand_Encoded'] = le_brand.fit_transform(df['Brand'])
print(df[['Brand', 'Brand_Encoded']])
print(dict(zip(le_brand.classes_, le_brand.transform(le_brand.classes_))))

**The Problem:** If we feed Brand_Encoded into a Linear Regression model, the model will mathematically interpret this as:

Toyota (2) = Honda (0) + Suzuki (1)

or that Toyota is "twice as much brand" as Suzuki — which is complete nonsense. There is no numerical relationship between car brands.

***Correct Fix: Use One-Hot Encoding instead for Brand and Fuel_Type:***

**Complete End-to-End Preprocessing Pipeline (Used Cars Dataset)**

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# 1. Load dataset
df = pd.read_csv('used_cars_dataset.csv')

# 2. Handle missing values first
df['Owner_Type'] = df['Owner_Type'].fillna(df['Owner_Type'].mode()[0])

# 3. Label Encode the ORDINAL column using manual mapping (preserves true order)
owner_mapping = {'First': 0, 'Second': 1, 'Third': 2, 'Fourth & Above': 3}
df['Owner_Encoded'] = df['Owner'].map(owner_mapping)

# 4. Label Encode the TARGET column (binary classification example: Transmission)
le_target = LabelEncoder()
df['Transmission_Encoded'] = le_target.fit_transform(df['Transmission'])

# 5. One-Hot Encode the NOMINAL input columns (correct choice for Brand, Fuel_Type, Seller_Type)
df = pd.get_dummies(df, columns=['Brand', 'Fuel_Type', 'Seller_Type'], drop_first=True)

# 6. Drop original text columns no longer needed as raw input
df = df.drop(columns=['Owner_Type', 'Transmission', 'Car_ID'])

print(df.head())
print(df.dtypes)